# 01 Collect Yearly Data

Canonical data collection notebook for the refactored AMR workflow.
This notebook builds yearly pair counts, yearly background marginals, runs the unified LLM screen, and exports `yearly_amr_screened_pairs.parquet`.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import (
    LLM_CLASSIFICATION_MAX_TOKENS,
    UNIFIED_AMR_INTEREST_SYSTEM_PROMPT,
    build_user_prompt,
    build_yearly_screened_pairs,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel
from amr.pipelines.collection import (
    build_collection_preflight_summary,
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)
from amr.pubmed.queries import (
    build_antibiotic_pathogen_query,
    build_antibiotic_query,
    build_pathogen_tiab_query,
)



## Glossary

- **AMR**: Antimicrobial Resistance.
- **SI**: Scientific Interest score, defined as residual AMR attention after background adjustment.
- **TIAB**: Title/Abstract PubMed field selector (search terms constrained to title or abstract text).
- **PMID**: PubMed Identifier for a publication record.
- **LLM**: Large Language Model used for abstract screening.
- **BG / background counts**: Yearly publication marginals for pathogens or antibiotics used in SI baseline modeling.
- **Slice**: A chosen year interval (`SLICE_START_YEAR` to `SLICE_END_YEAR`) processed in one run.


## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [3]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ASSETS_DIR = PROJECT_ROOT / "assets"
DATA_DIR = PROJECT_ROOT

PATHOGENS_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
PATHOGEN_PRIORITY_PATH = ASSETS_DIR / "counts_extended2.csv"
MIN_PATHOGENIC_INDEX = 0.0
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
OUTPUT_YEARLY_AMR_PATH = DATA_DIR / "candidate_amr_pair_year_counts.parquet"
OUTPUT_YEARLY_SCREEN_PATH = DATA_DIR / "yearly_amr_screened_pairs.parquet"
OUTPUT_UNIFIED_DECISIONS_PATH = DATA_DIR / "yearly_amr_unified_decisions.parquet"
OUTPUT_PATHOGEN_BG_PATH = DATA_DIR / "pathogen_tiab_counts.parquet"
OUTPUT_ANTIBIOTIC_BG_PATH = DATA_DIR / "antibiotic_tiab_counts.parquet"
INTERVAL_PMIDS_PATH = DATA_DIR / "pmids_results.parquet"
YEARLY_PAIR_PMID_PATH = DATA_DIR / "candidate_amr_pair_pmid_links.parquet"
PUBMED_METADATA_CACHE_PATH = DATA_DIR / "pubmed_pmid_metadata.parquet"
ABSTRACT_CACHE_PATH = DATA_DIR / "pmid_abstracts.parquet"
PUBMED_EMAIL = "xfcosta@gmail.com"  # replace with a real contact email

# Auto-resume behavior: all collection/filtering stages reuse on-disk outputs
# and continue only the missing work.

# Partition-friendly execution window: run a slice now, merge into canonical stores.
TOTAL_START_YEAR = 1970
TOTAL_END_YEAR = 2026
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = 2010
SLICE_END_YEAR = 2020
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = 5
NUM_PATHOGENS = 5  # use None for all pathogens; increase over time to expand cumulatively
NUM_ANTIBIOTICS = 5  # use None for all antibiotics; cache/resume skips previously completed work
# Existing notebook intent: AMR-focused retrieval before unified LLM relevance screen.
EXTRA_AMR_KEYWORDS = [
    "\"antimicrobial resistance\"[Title/Abstract] OR \"antibiotic resistance\"[Title/Abstract]"
]
RUN_LLM_FILTER = True
MAX_LLM_CALLS = None
# Editable per-run prompt (single source of truth for this notebook execution)
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
# Throughput/reliability controls for unified LLM filter
LLM_MAX_WORKERS = 2
LLM_BATCH_SIZE = 8
FAILED_RETRY_PATH = DATA_DIR / "yearly_amr_unified_failures.parquet"
MAX_FAILURE_RETRIES = 5
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True
# Batch-style collection logging controls
COLLECTION_VERBOSE = 1  # 0=silent, 1=batch summaries, 2=detailed per interval
COLLECTION_PROGRESS_EVERY = 300
PAIR_FULL_FETCH_THRESHOLD = 9500  # fetch once and partition locally below this PubMed count
BACKGROUND_SCOPE = "observed_entities"  # "observed_entities" is fast; "all_entities" is canonical/full
BACKGROUND_FETCH_THRESHOLD = 9500
BACKGROUND_METADATA_BATCH_SIZE = 200
LLM_CATEGORY_BACKFILL_DECISIONS = ("yes", "uncertain")  # prioritize SI-contributing rows; use None for all rows
# Unified filter logging cadence
UNIFIED_PROGRESS_EVERY = 200



In [4]:
from textwrap import fill

print("PRETTY PRINT (80 COLS) OF UNIFIED_SYSTEM_PROMPT:\n")
print("\n".join(fill(line, width=80) if line.strip() else "" for line in UNIFIED_SYSTEM_PROMPT.splitlines()))


PRETTY PRINT (80 COLS) OF UNIFIED_SYSTEM_PROMPT:

You are classifying scientific abstracts about antimicrobial resistance (AMR).

Task:
Given a target pathogen, a target antibiotic or antibiotic class, a title, and
an abstract,
classify the abstract into exactly one primary category describing why this
paper contributes
to scientific interest for the target pathogen-antibiotic pair.

Categories:

A. RESISTANCE_PREVALENCE_SURVEILLANCE
Use this category when the abstract mainly reports resistance rates,
susceptibility frequencies,
incidence, prevalence, temporal trends, geographic spread, epidemiology,
outbreak burden,
surveillance results, or antibiogram-style findings for the target pathogen-
antibiotic pair.

B. RESISTANCE_MECHANISM
Use this category when the abstract mainly studies biological or molecular
mechanisms of
resistance for the target pair, including resistance genes, beta-lactamases,
carbapenemases,
ESBLs, KPC, NDM, OXA, CTX-M, plasmids, integrons, transposons, horizontal 

## Load Prepared Pathogen and Antibiotic Inputs



In [5]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_csv(ANTIBIOTIC_COARSENING_PATH)
antibiotic_synonym_df = pd.read_csv(ANTIBIOTIC_SYNONYM_PATH)

if not PATHOGEN_PRIORITY_PATH.exists():
    raise FileNotFoundError(
        f"Pathogen priority file not found: {PATHOGEN_PRIORITY_PATH}. "
        "Run notebooks/00_prepare_inputs.ipynb first."
    )

pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_PATH)
required_priority_cols = {"CanonicalSpecies", "pathogenic_index"}
missing_priority_cols = required_priority_cols - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_PATH} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"],
    errors="coerce",
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[
    pathogen_priority_df["pathogenic_index"] > float(MIN_PATHOGENIC_INDEX)
]
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"],
    ascending=[False, True],
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")

all_pathogens = pathogen_priority_df["CanonicalSpecies"].tolist()
all_antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

pathogens = all_pathogens if NUM_PATHOGENS is None else all_pathogens[: int(NUM_PATHOGENS)]
antibiotics = all_antibiotics if NUM_ANTIBIOTICS is None else all_antibiotics[: int(NUM_ANTIBIOTICS)]

print(f"Pathogen priority source: {PATHOGEN_PRIORITY_PATH}")
print(f"Full synonym lookup vocabulary: {bacteria_synonym_df['bacteria name'].nunique():,}")
print(
    f"Pathogenic-index pathogen vocabulary: {len(all_pathogens):,} "
    f"(MIN_PATHOGENIC_INDEX>{MIN_PATHOGENIC_INDEX})"
)
print(f"Antibiotic coarsening source: {ANTIBIOTIC_COARSENING_PATH}")
print(f"Antibiotic synonym source: {ANTIBIOTIC_SYNONYM_PATH} ({len(antibiotic_synonym_df):,} rows)")
print(f"Full antibiotic vocabulary: {len(all_antibiotics):,}")
print(f"Active pathogens this run: {len(pathogens):,} (NUM_PATHOGENS={NUM_PATHOGENS})")
print(f"Active antibiotics this run: {len(antibiotics):,} (NUM_ANTIBIOTICS={NUM_ANTIBIOTICS})")
print("Top active pathogens by pathogenic_index:")
display(
    pathogen_priority_df[["CanonicalSpecies", "pathogenic_index", "n_causation", "n_articles"]]
    .head(len(pathogens))
    .reset_index(drop=True)
)



Pathogen priority source: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/counts_extended2.csv
Full synonym lookup vocabulary: 19,956
Pathogenic-index pathogen vocabulary: 915 (MIN_PATHOGENIC_INDEX>0.0)
Antibiotic coarsening source: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv
Antibiotic synonym source: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv (422 rows)
Full antibiotic vocabulary: 245
Active pathogens this run: 5 (NUM_PATHOGENS=5)
Active antibiotics this run: 5 (NUM_ANTIBIOTICS=5)
Top active pathogens by pathogenic_index:


,CanonicalSpecies,pathogenic_index,n_causation,n_articles
0,Salmonella enterica,1.000000,554,24694
1,Mycobacterium tuberculosis,0.928144,299,20559
2,Clostridioides difficile,0.910833,298,17012
3,Haemophilus influenzae,0.860892,200,14537
4,Klebsiella pneumoniae,0.857160,176,15847


## Collect Yearly Pair Dataset and Background Marginals


## PubMed Query Logic for Abstract Collection

The collection query is assembled per `(pathogen, antibiotic)` pair with four components:

1. **Antibiotic clause**: antibiotic name in `Abstract`, including split variants for hyphen/slash names (for example `co-trimoxazole` as exact phrase OR split terms).
2. **Pathogen clause**: pathogen + known synonyms in `Abstract` (`OR`-joined).
3. **AMR clause**: additional AMR-focused condition from `EXTRA_AMR_KEYWORDS` in `Title/Abstract`.
4. **Date clause**: publication date restricted to `[MIN_DATE, MAX_DATE]`.

Final query shape:

`(antibiotic_clause AND pathogen_clause) AND extra_amr_clause AND date_range_clause`


In [6]:
sample_pathogen = pathogens[0]
sample_antibiotic = antibiotics[0]
sample_query = build_antibiotic_pathogen_query(
    sample_antibiotic,
    sample_pathogen,
    bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    extra_clauses=EXTRA_AMR_KEYWORDS,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
)
print("Sample pathogen:", sample_pathogen)
print("Sample antibiotic:", sample_antibiotic)
print("\nSample PubMed query:\n")
print(sample_query)


Sample pathogen: Salmonella enterica
Sample antibiotic: Amikacin

Sample PubMed query:

(("Amikacin"[Abstract]) AND ("Salmonella enterica"[Abstract] OR "salmonella arizonae"[Abstract] OR "salmonella choleraesuis"[Abstract] OR "salmonella enterica"[Abstract] OR "salmonella enteritidis"[Abstract] OR "salmonella paratyphi"[Abstract] OR "salmonella typhi"[Abstract] OR "salmonella typhimurium"[Abstract])) AND ("antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract]) AND ("2010/01/01"[Date - Publication] : "2020/12/31"[Date - Publication])


In [7]:
print("EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)\n")

example_pairs = list(zip(pathogens[:3], antibiotics[:3]))
for idx, (p_name, a_name) in enumerate(example_pairs, start=1):
    pathogen_clause = build_pathogen_tiab_query(p_name, bacteria_synonym_df)
    antibiotic_clause = build_antibiotic_query(a_name, antibiotic_synonym_df, field="Title/Abstract")
    pair_query = build_antibiotic_pathogen_query(
        a_name,
        p_name,
        bacteria_synonym_df,
        antibiotic_synonym_df=antibiotic_synonym_df,
        extra_clauses=EXTRA_AMR_KEYWORDS,
        min_date=MIN_DATE,
        max_date=MAX_DATE,
    )
    print(f"Example {idx}: {p_name} vs {a_name}")
    print("- Pathogen TIAB clause:")
    print(pathogen_clause)
    print("- Antibiotic TIAB clause:")
    print(antibiotic_clause)
    print("- Final pair + AMR + date query:")
    print(pair_query)
    print("\n" + "-" * 100 + "\n")


EXAMPLE PUBMED QUERIES (MULTIPLE VIEWS)

Example 1: Salmonella enterica vs Amikacin
- Pathogen TIAB clause:
("Salmonella enterica"[Title/Abstract] OR "salmonella arizonae"[Title/Abstract] OR "salmonella choleraesuis"[Title/Abstract] OR "salmonella enterica"[Title/Abstract] OR "salmonella enteritidis"[Title/Abstract] OR "salmonella paratyphi"[Title/Abstract] OR "salmonella typhi"[Title/Abstract] OR "salmonella typhimurium"[Title/Abstract])
- Antibiotic TIAB clause:
("Amikacin"[Title/Abstract])
- Final pair + AMR + date query:
(("Amikacin"[Abstract]) AND ("Salmonella enterica"[Abstract] OR "salmonella arizonae"[Abstract] OR "salmonella choleraesuis"[Abstract] OR "salmonella enterica"[Abstract] OR "salmonella enteritidis"[Abstract] OR "salmonella paratyphi"[Abstract] OR "salmonella typhi"[Abstract] OR "salmonella typhimurium"[Abstract])) AND ("antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract]) AND ("2010/01/01"[Date - Publication] : "2020/12/31"[Date - P

In the next code cell, the notebook will automatically gather research paper IDs from PubMed for each pathogen-antibiotic combination in your selected year range, convert those papers into yearly counts, and then save the results to local files so future runs can continue from where you left off. It also collects yearly background publication totals for each pathogen and antibiotic, which are later used to calculate scientific interest in a fair, comparable way.


## Resume Preflight

Estimate how much work is already cached on disk before the resumable collection step starts.


In [8]:
preflight = build_collection_preflight_summary(
    pathogens=pathogens,
    antibiotics=antibiotics,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_path=YEARLY_PAIR_PMID_PATH,
    yearly_output_path=OUTPUT_YEARLY_AMR_PATH,
    pathogen_bg_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_bg_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    background_start_year=SLICE_START_YEAR,
    background_end_year=SLICE_END_YEAR,
    total_min_date=TOTAL_MIN_DATE,
    total_max_date=TOTAL_MAX_DATE,
    total_background_start_year=TOTAL_START_YEAR,
    total_background_end_year=TOTAL_END_YEAR,
)

def _print_window_report(label: str, window_summary: dict) -> None:
    interval_cache = window_summary["interval_cache"]
    pathogen_bg_cache = window_summary["pathogen_bg_cache"]
    antibiotic_bg_cache = window_summary["antibiotic_bg_cache"]
    print(label)
    print(f"Date window: {window_summary['min_date']} .. {window_summary['max_date']}")
    print(f"Intervals in window: {len(window_summary['intervals'])} -> {window_summary['intervals']}")
    print(
        f"Interval cache coverage: {interval_cache['cached_rows']:,} / {interval_cache['expected_rows']:,} "
        f"rows ({interval_cache['cached_pct']:.2f}%)"
    )
    print(f"Missing interval rows still to collect: {interval_cache['missing_rows']:,}")
    print(
        f"Cached interval breadth: {interval_cache['unique_pathogens']:,} pathogens, "
        f"{interval_cache['unique_antibiotics']:,} antibiotics, "
        f"{interval_cache['unique_pairs']:,} unique pairs"
    )
    print(f"Existing yearly AMR rows in window: {window_summary['yearly_rows']:,}")
    print(f"Existing yearly pair PMID rows in window: {window_summary['yearly_pair_pmid_rows']:,}")
    print(
        f"Pathogen background cache: {pathogen_bg_cache['cached_rows']:,} / "
        f"{pathogen_bg_cache['expected_rows']:,} rows ({pathogen_bg_cache['cached_pct']:.2f}%)"
    )
    pathogen_missing_pct = (100.0 - pathogen_bg_cache['cached_pct']) if pathogen_bg_cache['expected_rows'] else 0.0
    print(
        f"Missing pathogen background rows: {pathogen_bg_cache['missing_rows']:,} "
        f"({pathogen_missing_pct:.2f}%)"
    )
    if pathogen_bg_cache['warning']:
        print(f"WARNING: {pathogen_bg_cache['warning']}")
    print(
        f"Antibiotic background cache: {antibiotic_bg_cache['cached_rows']:,} / "
        f"{antibiotic_bg_cache['expected_rows']:,} rows ({antibiotic_bg_cache['cached_pct']:.2f}%)"
    )
    antibiotic_missing_pct = (100.0 - antibiotic_bg_cache['cached_pct']) if antibiotic_bg_cache['expected_rows'] else 0.0
    print(
        f"Missing antibiotic background rows: {antibiotic_bg_cache['missing_rows']:,} "
        f"({antibiotic_missing_pct:.2f}%)"
    )
    if antibiotic_bg_cache['warning']:
        print(f"WARNING: {antibiotic_bg_cache['warning']}")
    print()

print("COLLECTION PREFLIGHT\n")
print(f"Existing yearly AMR span on disk: {preflight['yearly_year_min']} .. {preflight['yearly_year_max']}")
print(f"Existing yearly AMR rows on disk: {preflight['yearly_rows']:,}")
print(f"Existing yearly pair PMID rows on disk: {preflight['yearly_pair_pmid_rows']:,}\n")
print("TOTAL WINDOW SUMMARY")
_print_window_report("", preflight['total_window'])
print("CURRENT SLICE SUMMARY")
_print_window_report("", preflight['run_window'])

if preflight['run_window']['interval_cache']['missing_rows'] > 0:
    print("WARNING: collection is resumable, but a non-trivial amount of interval PubMed work remains for the current slice.")
    if preflight['run_window']['interval_cache']['cached_pct'] < 50:
        print("Current-slice interval cache coverage is below 50%, so this run may still be expensive.")


COLLECTION PREFLIGHT

Existing yearly AMR span on disk: 2010 .. 2026
Existing yearly AMR rows on disk: 242
Existing yearly pair PMID rows on disk: 3,045

TOTAL WINDOW SUMMARY

Date window: 1970/01/01 .. 2026/12/31
Intervals in window: 12 -> [('1970/01/01', '1974/12/31'), ('1975/01/01', '1979/12/31'), ('1980/01/01', '1984/12/31'), ('1985/01/01', '1989/12/31'), ('1990/01/01', '1994/12/31'), ('1995/01/01', '1999/12/31'), ('2000/01/01', '2004/12/31'), ('2005/01/01', '2009/12/31'), ('2010/01/01', '2014/12/31'), ('2015/01/01', '2019/12/31'), ('2020/01/01', '2024/12/31'), ('2025/01/01', '2026/12/31')]
Interval cache coverage: 100 / 300 rows (33.33%)
Missing interval rows still to collect: 200
Cached interval breadth: 5 pathogens, 5 antibiotics, 25 unique pairs
Existing yearly AMR rows in window: 242
Existing yearly pair PMID rows in window: 3,045
Pathogen background cache: 59 / 285 rows (20.70%)
Missing pathogen background rows: 226 (79.30%)
Antibiotic background cache: 35 / 285 rows (12.28%)

In [9]:
print("Auto-resume: collection stages will reuse saved outputs and continue missing work only.")

yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
    full_fetch_threshold=PAIR_FULL_FETCH_THRESHOLD,
    collection_verbose=COLLECTION_VERBOSE,
    collection_progress_every=COLLECTION_PROGRESS_EVERY,
)


Auto-resume: collection stages will reuse saved outputs and continue missing work only.
Collection summary: events=75, cached=75, zero=0, full_fetch=0, new=0
PubMed metadata cache already covers all 0 requested PMIDs.


In [ ]:
if BACKGROUND_SCOPE == "observed_entities":
    background_pathogens = sorted(yearly_amr_df["pathogen"].dropna().astype(str).unique())
    background_antibiotics = sorted(yearly_amr_df["antibiotic"].dropna().astype(str).unique())
elif BACKGROUND_SCOPE == "all_entities":
    background_pathogens = pathogens
    background_antibiotics = antibiotics
else:
    raise ValueError("BACKGROUND_SCOPE must be 'observed_entities' or 'all_entities'.")

print(
    f"Background scope={BACKGROUND_SCOPE}: "
    f"{len(background_pathogens):,} pathogens, {len(background_antibiotics):,} antibiotics"
)

pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=background_pathogens,
    antibiotics=background_antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    fetch_threshold=BACKGROUND_FETCH_THRESHOLD,
    metadata_batch_size=BACKGROUND_METADATA_BATCH_SIZE,
)

print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"yearly pair PMID rows: {len(yearly_pair_pmid_df):,}")
print(f"newly enriched pair PMID rows this run: {len(enriched_pair_pmid_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")


Background scope=observed_entities: 5 pathogens, 5 antibiotics
[search_term_clause] starting query window 2010/01/01..2019/12/31; max attempts=3


## Unified LLM Filtering
Single strict pass: AMR evidence + scientific-interest relevance.


## Unified LLM Prompt Templates
This workflow uses a single strict LLM screen:
- **System prompt**: policy for AMR evidence + scientific-interest relevance.
- **User prompt template**: PMID-specific payload (pathogen, antibiotic, title, abstract).
**Prompt governance note:** if you change this prompt between slices, document it so downstream results remain interpretable.


In [ ]:
print("UNIFIED SYSTEM PROMPT:\n")
print(UNIFIED_SYSTEM_PROMPT)
print("\n" + "=" * 80 + "\n")
print("USER PROMPT TEMPLATE EXAMPLE:\n")
print(
    build_user_prompt(
        pmid="12345678",
        pathogen="Escherichia coli",
        antibiotic="Ciprofloxacin",
        title="Template title",
        abstract="Template abstract text.",
    )
)


In [ ]:
candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")

resume_df = None
if OUTPUT_UNIFIED_DECISIONS_PATH.exists():
    resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
    print(f"Found existing unified decisions: {len(resume_df):,} rows. Missing candidates will be resumed automatically.")

if RUN_LLM_FILTER:
    llm = OpenAILanguageModel(OpenAIConfig(model="gpt-4o-mini", max_tokens=LLM_CLASSIFICATION_MAX_TOKENS, temperature=0))
    unified_df = run_unified_llm_filter(
        candidate_df,
        llm,
        email=PUBMED_EMAIL,
        system_prompt=UNIFIED_SYSTEM_PROMPT,
        output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
        resume_from=resume_df,
        auto_resume_from_output=True,
        llm_max_workers=LLM_MAX_WORKERS,
        llm_batch_size=LLM_BATCH_SIZE,
        abstract_fetch_batch_size=ABSTRACT_FETCH_BATCH_SIZE,
        prefetch_abstracts=PREFETCH_ABSTRACTS,
        failed_retry_path=str(FAILED_RETRY_PATH),
        max_failure_retries=MAX_FAILURE_RETRIES,
        abstract_cache_path=str(ABSTRACT_CACHE_PATH),
        max_llm_calls=MAX_LLM_CALLS,
        category_backfill_decisions=LLM_CATEGORY_BACKFILL_DECISIONS,
        progress_every=UNIFIED_PROGRESS_EVERY,
        verbose=1,
    )
else:
    if resume_df is None or resume_df.empty:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Enable RUN_LLM_FILTER to create it."
        )
    unified_df = resume_df
    print(f"Loaded unified decisions from {OUTPUT_UNIFIED_DECISIONS_PATH} ({len(unified_df):,} rows).")

print(unified_df["unified_decision"].value_counts(dropna=False))
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True


## Export Canonical Screened Yearly Dataset
Includes decisions `yes` + `uncertain` by default.


In [ ]:
yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    unified_df,
    include_decisions=("yes", "uncertain"),
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=True,
)

print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {yearly_screened_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
print(f"Year span: {int(yearly_screened_df['year'].min())} - {int(yearly_screened_df['year'].max())}")
yearly_screened_df.head(10)

